In [ ]:
import os
from pathlib import Path

get_ipython().system("nvidia-smi")

repo_url = "https://github.com/Sajin2404/GeneWeaver.git"
repo_dir = Path("/content/GeneWeaver")

if not repo_dir.exists():
    get_ipython().system(f"git clone {repo_url} {repo_dir}")
else:
    get_ipython().system(f"git -C {repo_dir} pull origin main")

os.chdir(repo_dir)
print("Working dir:", os.getcwd())

In [ ]:
import time
import numpy as np

from alignment.gpu_aligner import align_distributed, is_cuda_available, run_gpu_alignment
from benchmarks.record_benchmark import record_and_export

assert is_cuda_available(), "CUDA GPU not available on this runtime."

mapping = {"A": 0, "C": 1, "G": 2, "T": 3}
grna_str = "ACGTACGTACGTACGTACGT"
seq_chars = ["A"] * 1_000_000
seq_chars[150_000 : 150_000 + len(grna_str)] = list(grna_str)
seq_str = "".join(seq_chars)

seq_enc = np.array([mapping[c] for c in seq_str], dtype=np.uint8)
grna_enc = np.array([mapping[c] for c in grna_str], dtype=np.uint8)

single_hits, single_time, _ = run_gpu_alignment(seq_enc, grna_enc, max_mismatches=2)

t0 = time.perf_counter()
dist_out = align_distributed(seq_enc, grna_enc, max_mismatches=2, partitions_per_worker=1)
dist_time = time.perf_counter() - t0

dist_hits = [(h["position"], h["mismatches"]) for h in dist_out["hits"]]
assert dist_hits == single_hits, f"Mismatch: single={single_hits}, dist={dist_hits}"

run_id, _ = record_and_export(
    environment="Colab GPU (VS Code)",
    device_name="NVIDIA T4",
    execution_mode="Dask Distributed GPU",
    sequence_length=len(seq_enc),
    grna_seq=grna_str,
    max_mismatches=2,
    hits=dist_hits,
    gpu_time_sec=dist_time,
    chunk_size=len(seq_enc) // max(1, dist_out["chunks_processed"]),
    notes=f"Single-GPU: {single_time:.4f}s | Workers: {dist_out['workers_used']} | Chunks: {dist_out['chunk_summaries']}",
)

print(f"Parity OK! Hits={dist_hits}, Single={single_time:.4f}s, Dist={dist_time:.4f}s, run_id={run_id}")